<a href="https://colab.research.google.com/github/JulianSantos-LATAMAI/ECON-5200/blob/main/Lab_12/%5BLab_12%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

In [3]:
# Step 1: Ingestion from external source
df = pd.read_csv(url)


# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'


# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula,data=df)
results = model.fit()
print(results.summary())


                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Wed, 11 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        11:36:43   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [4]:

# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)


# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")



The Predictive RMSE is: $42,316.69


In [5]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go

# ── 0. REPRODUCIBLE SAMPLE DATA ──────────────────────────────────────────────
# Replace this block with your actual df, X, and y from your lab.
np.random.seed(42)
n = 200
df = pd.DataFrame({
    "sqft":    np.random.randint(500, 4000, n),
    "bedrooms": np.random.randint(1, 6, n),
    "age":     np.random.randint(0, 50, n),
    "distance": np.random.uniform(0.5, 30, n),
})
df["price"] = (
    100 * df["sqft"]
    + 15000 * df["bedrooms"]
    - 500 * df["age"]
    - 3000 * df["distance"]
    + np.random.normal(0, 20000, n)
)

# ── 1. FIT THE OLS MODEL ──────────────────────────────────────────────────────
X = sm.add_constant(df[["sqft", "bedrooms", "age", "distance"]])  # adds intercept
y = df["price"]

model   = sm.OLS(y, X)
results = model.fit()
print(results.summary())

# ── 2. EXTRACT RESIDUALS & FITTED VALUES FROM STATSMODELS ─────────────────────
# results.fittedvalues → Series of ŷ (predicted values) indexed like y
# results.resid        → Series of (y - ŷ) raw residuals, same index
fitted    = results.fittedvalues
residuals = results.resid

# ── 3. RMSE ───────────────────────────────────────────────────────────────────
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: ${rmse:,.2f}")

# ── 4. FLAG OUTLIERS (|residual| > 2 × std) ───────────────────────────────────
resid_std   = residuals.std()           # 1 SD of the residual distribution
threshold   = 2 * resid_std             # 2-SD boundary

# Boolean mask: True where the absolute residual exceeds the threshold
is_outlier  = residuals.abs() > threshold

# Build a tidy DataFrame for Plotly — always cleaner than passing raw Series
plot_df = pd.DataFrame({
    "fitted":    fitted,
    "residuals": residuals,
    # Color label used by px.scatter's color= argument
    "point_type": np.where(is_outlier, "Outlier (>2 SD)", "Normal"),
    # Hover label shows the original observation index
    "obs_index": residuals.index,
})

# ── 5. BUILD THE FIGURE ───────────────────────────────────────────────────────
COLOR_MAP = {
    "Normal":          "#4A90D9",   # steel blue  — well-behaved residuals
    "Outlier (>2 SD)": "#DC143C",   # crimson     — flagged observations
}

fig = px.scatter(
    plot_df,
    x="fitted",
    y="residuals",
    color="point_type",
    color_discrete_map=COLOR_MAP,
    hover_data={"obs_index": True, "fitted": ":.2f", "residuals": ":.2f"},
    labels={
        "fitted":     "Fitted Values (ŷ)",
        "residuals":  "Residuals (y − ŷ)",
        "point_type": "Point Type",
    },
    title=(
        f"Residual Forensics Dashboard — Hedonic Pricing OLS<br>"
        f"<sup>RMSE: ${rmse:,.0f} | Outliers flagged at ±{threshold:,.0f} "
        f"({is_outlier.sum()} points)</sup>"
    ),
    opacity=0.75,
    template="plotly_dark",
)

# ── 6. ZERO-LINE & SD BANDS ───────────────────────────────────────────────────
x_range = [fitted.min(), fitted.max()]

# Horizontal zero-line: visually separates over- from under-predictions
fig.add_shape(
    type="line", x0=x_range[0], x1=x_range[1], y0=0, y1=0,
    line=dict(color="white", width=1.5, dash="solid"),
)

# ±2 SD reference bands — anything outside these is crimson
for sign in [1, -1]:
    fig.add_shape(
        type="line",
        x0=x_range[0], x1=x_range[1],
        y0=sign * threshold, y1=sign * threshold,
        line=dict(color="#DC143C", width=1, dash="dash"),
    )

# Shaded band between ±2 SD to give spatial context
fig.add_shape(
    type="rect",
    x0=x_range[0], x1=x_range[1],
    y0=-threshold,  y1=threshold,
    fillcolor="rgba(74,144,217,0.07)",
    line_width=0,
    layer="below",
)

fig.update_layout(
    xaxis_title="Fitted Values (ŷ)",
    yaxis_title="Residuals (y − ŷ)",
    legend_title_text="Point Type",
    font=dict(size=13),
    height=580,
    margin=dict(t=100),
)

fig.show()

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.953
Model:                            OLS   Adj. R-squared:                  0.952
Method:                 Least Squares   F-statistic:                     997.5
Date:                Wed, 11 Mar 2026   Prob (F-statistic):          1.37e-128
Time:                        11:41:32   Log-Likelihood:                -2290.1
No. Observations:                 200   AIC:                             4590.
Df Residuals:                     195   BIC:                             4607.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       2558.0186   6915.433      0.370      0.7